<div dir="rtl">

# جلسه ۲: پس‌انتشار خطا و گرادیان کاهشی از صفر

**Backpropagation and Gradient Descent from Scratch**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/keyvanmahmoudi/Deep_Learning_Course/blob/main/02.%20Backpropagation_and_Gradient_Descent.ipynb)

**دوره یادگیری عمیق و شبکه‌های عصبی** — کیوان محمودی

### اهداف این جلسه

- آشنایی با توابع هزینه (Loss Functions): MSE و Cross-Entropy
- درک گرادیان کاهشی (Gradient Descent) و اثر نرخ یادگیری
- ساخت یک موتور مشتق‌گیری خودکار کوچک روی گراف محاسباتی (Computational Graph)
- استخراج و پیاده‌سازی Backpropagation برای MLP به فرم ماتریسی با NumPy
- بررسی درستی گرادیان با Gradient Checking و مشاهده پدیده Vanishing Gradient

</div>

<div dir="rtl">

## ۱. مرور: آموزش یعنی کمینه‌سازی تابع هزینه

در جلسه قبل یک MLP ساختیم که وزن‌هایش تصادفی بود. **آموزش (Training)** یعنی پیدا کردن وزن‌هایی که خطای مدل روی داده آموزشی را کمینه کنند. برای این کار به سه جزء نیاز داریم:

۱. **تابع هزینه (Loss Function)** $\mathcal{L}(\theta)$ که میزان خطا را با یک عدد بیان کند.

۲. **گرادیان (Gradient)** $\nabla_\theta \mathcal{L}$ که جهت بیشترین افزایش هزینه را نشان دهد.

۳. **الگوریتم بهینه‌سازی** که با استفاده از گرادیان، پارامترها را به‌روز کند.

**پس‌انتشار خطا (Backpropagation)** الگوریتمی برای محاسبه کارآمد گرادیان نسبت به **همه** پارامترهای شبکه است.

</div>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

np.random.seed(0)
plt.rcParams['figure.figsize'] = (6, 4)

<div dir="rtl">

## ۲. توابع هزینه (Loss Functions)

**میانگین مربعات خطا (Mean Squared Error)** برای رگرسیون:

$$\mathcal{L}_{\text{MSE}} = \frac{1}{N}\sum_{i=1}^{N}\left(y_i - \hat{y}_i\right)^2$$

**آنتروپی متقاطع (Cross-Entropy)** برای طبقه‌بندی. اگر $\mathbf{y}$ بردار One-hot برچسب واقعی و $\hat{\mathbf{y}}$ خروجی Softmax باشد:

$$\mathcal{L}_{\text{CE}} = -\frac{1}{N}\sum_{i=1}^{N}\sum_{k=1}^{K} y_{ik}\log \hat{y}_{ik} = -\frac{1}{N}\sum_{i=1}^{N}\log \hat{y}_{i,\,c_i}$$

یعنی فقط احتمالی که مدل به **کلاس درست** $c_i$ داده مهم است. Cross-Entropy همان منفی لگاریتم درست‌نمایی (Negative Log-Likelihood) است: کمینه کردن آن یعنی بیشینه کردن احتمال داده‌های مشاهده‌شده تحت مدل (Maximum Likelihood Estimation).

چرا برای طبقه‌بندی از MSE استفاده نمی‌کنیم؟ وقتی مدل با اطمینان زیاد اشتباه می‌کند، Cross-Entropy جریمه بسیار بزرگی می‌دهد (به سمت بی‌نهایت میل می‌کند)، اما MSE حداکثر جریمه‌ای برابر ۱ می‌دهد:

</div>

In [ ]:
p = np.linspace(0.001, 1, 500)          # probability assigned to the correct class
plt.plot(p, -np.log(p), lw=2, label='Cross-Entropy: $-\\log p$')
plt.plot(p, (1 - p) ** 2, lw=2, label='MSE: $(1-p)^2$')
plt.xlabel('probability of the correct class'); plt.ylabel('loss')
plt.ylim(0, 5); plt.legend(); plt.title('Penalty for a confident mistake'); plt.show()

<div dir="rtl">

## ۳. گرادیان کاهشی (Gradient Descent)

گرادیان جهت **بیشترین افزایش** تابع است؛ پس برای کاهش هزینه، خلاف جهت گرادیان قدم برمی‌داریم:

$$\theta \leftarrow \theta - \eta\,\nabla_\theta \mathcal{L}(\theta)$$

انتخاب **نرخ یادگیری** $\eta$ حیاتی است. روی تابع ساده $f(\theta) = \theta^2$ (با مشتق $2\theta$) آزمایش می‌کنیم:

</div>

In [ ]:
def gradient_descent_1d(grad, theta0, lr, steps=15):
    path = [theta0]
    for _ in range(steps):
        path.append(path[-1] - lr * grad(path[-1]))
    return np.array(path)


f = lambda t: t ** 2
df = lambda t: 2 * t
grid = np.linspace(-3.2, 3.2, 200)

fig, axes = plt.subplots(1, 4, figsize=(18, 3.8), sharey=True)
for ax, lr, title in zip(axes, [0.05, 0.3, 0.9, 1.05], ['too small', 'good', 'oscillating', 'diverging']):
    path = gradient_descent_1d(df, 2.8, lr)
    ax.plot(grid, f(grid), color='lightgray', lw=3)
    ax.plot(path, f(path), 'o-', ms=5)
    ax.set_title(f'lr = {lr} ({title})'); ax.set_ylim(-0.5, 11)
plt.show()

<div dir="rtl">

- نرخ یادگیری **کوچک**: همگرایی بسیار کند.
- نرخ یادگیری **مناسب**: همگرایی سریع.
- نرخ یادگیری **بزرگ**: نوسان یا حتی **واگرایی (Divergence)**.

در شبکه‌های عصبی، تابع هزینه میلیون‌ها پارامتر دارد و **محدب (Convex)** نیست؛ اما همین ایده ساده موتور اصلی آموزش همه مدل‌های یادگیری عمیق است.

## ۴. گراف محاسباتی و قاعده زنجیره‌ای (Chain Rule)

هر محاسبه در شبکه عصبی را می‌توان به صورت یک **گراف محاسباتی (Computational Graph)** از عملیات ساده (جمع، ضرب، تابع فعال‌سازی) نوشت. برای مثال $L = (a\cdot b + c)^2$:

- **Forward:** مقدار هر گره را از ورودی به خروجی حساب می‌کنیم: $d = a\cdot b$، $e = d + c$، $L = e^2$.
- **Backward:** مشتق $L$ نسبت به هر گره را از خروجی به ورودی با **قاعده زنجیره‌ای** حساب می‌کنیم:

$$\frac{\partial L}{\partial a} = \frac{\partial L}{\partial e}\cdot\frac{\partial e}{\partial d}\cdot\frac{\partial d}{\partial a} = 2e \cdot 1 \cdot b$$

نکته کلیدی: هر گره فقط باید **مشتق محلی (Local Gradient)** خودش را بداند. گرادیانی که از بالا می‌رسد (**Upstream Gradient**) در مشتق محلی ضرب می‌شود و به گره‌های پایین‌تر فرستاده می‌شود. اگر یک گره در چند مسیر استفاده شده باشد، گرادیان‌های رسیده از مسیرها **جمع** می‌شوند.

بیایید یک موتور مشتق‌گیری خودکار (Automatic Differentiation) کوچک بسازیم؛ این دقیقاً همان کاری است که `autograd` در PyTorch انجام می‌دهد (ایده از پروژه micrograd، آندری کارپاتی):

</div>

In [ ]:
class Value:
    """A scalar that remembers how it was computed, so it can backpropagate gradients."""

    def __init__(self, data, parents=(), op=''):
        self.data = float(data)
        self.grad = 0.0
        self._parents = parents
        self._backward = lambda: None
        self.op = op

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        def _backward():
            self.grad += out.grad            # d(a+b)/da = 1
            other.grad += out.grad           # d(a+b)/db = 1
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        def _backward():
            self.grad += other.data * out.grad   # d(ab)/da = b
            other.grad += self.data * out.grad   # d(ab)/db = a
        out._backward = _backward
        return out

    def __pow__(self, k):
        out = Value(self.data ** k, (self,), f'**{k}')
        def _backward():
            self.grad += k * self.data ** (k - 1) * out.grad
        out._backward = _backward
        return out

    def relu(self):
        out = Value(max(0.0, self.data), (self,), 'relu')
        def _backward():
            self.grad += (out.data > 0) * out.grad
        out._backward = _backward
        return out

    def backward(self):
        # topological order: every node is processed after all nodes that depend on it
        order, seen = [], set()
        def build(v):
            if v not in seen:
                seen.add(v)
                for p in v._parents:
                    build(p)
                order.append(v)
        build(self)
        self.grad = 1.0
        for v in reversed(order):
            v._backward()

    __radd__ = __add__
    __rmul__ = __mul__

    def __repr__(self):
        return f'Value(data={self.data:.4f}, grad={self.grad:.4f})'

In [ ]:
a, b, c = Value(2.0), Value(-3.0), Value(10.0)
d = a * b          # -6
e = d + c          #  4
L = e ** 2         # 16
L.backward()

print('L =', L.data)
print('dL/da =', a.grad, '  expected 2e*b =', 2 * 4 * -3)
print('dL/db =', b.grad, '  expected 2e*a =', 2 * 4 * 2)
print('dL/dc =', c.grad, '  expected 2e   =', 2 * 4)

<div dir="rtl">

با همین کلاس کوچک می‌توان یک نورون را آموزش داد. نورون زیر باید یاد بگیرد برای ورودی $(x_1, x_2) = (1, 2)$ خروجی $3$ بدهد:

</div>

In [ ]:
w1, w2, bias = Value(0.3), Value(0.2), Value(0.0)
x1, x2, target = 1.0, 2.0, 3.0

for step in range(30):
    y_hat = (w1 * x1 + w2 * x2 + bias).relu()
    loss = (y_hat + (-target)) ** 2
    for p in (w1, w2, bias):        # gradients accumulate, so reset them before each backward pass
        p.grad = 0.0
    loss.backward()
    for p in (w1, w2, bias):        # gradient descent step
        p.data -= 0.05 * p.grad
    if step % 5 == 0 or step == 29:
        print(f'step {step:2d}  loss = {loss.data:.5f}  y_hat = {y_hat.data:.4f}')

<div dir="rtl">

**دقت کنید:** به دلیل `+=` در توابع backward، گرادیان‌ها **انباشته (Accumulate)** می‌شوند؛ پس قبل از هر backward باید آن‌ها را صفر کنیم. در PyTorch هم دقیقاً به همین دلیل `optimizer.zero_grad()` را صدا می‌زنیم.

یک آزمایش: آموزش را با وزن‌های اولیه منفی تکرار کنید (مثلاً `w1, w2 = Value(-1.0), Value(-1.0)`). چون $z$ منفی می‌شود، خروجی و گرادیان ReLU صفر است و نورون هیچ‌وقت یاد نمی‌گیرد؛ این همان مشکل **نورون مرده (Dead Neuron)** است.

## ۵. Backpropagation برای MLP به فرم ماتریسی

موتور اسکالر بالا برای فهم مفهوم عالی است اما بسیار کند است. در عمل، محاسبات را روی **ماتریس‌ها** انجام می‌دهیم. یک شبکه دولایه با ورودی $X$ (ابعاد $N\times d$) را در نظر بگیرید:

$$Z_1 = XW_1 + \mathbf{b}_1,\quad H = \text{ReLU}(Z_1),\quad Z_2 = HW_2 + \mathbf{b}_2,\quad P = \text{softmax}(Z_2)$$

و هزینه Cross-Entropy. با قاعده زنجیره‌ای، گرادیان‌ها از آخر به اول به دست می‌آیند ($Y$ ماتریس One-hot برچسب‌ها است):

$$\frac{\partial \mathcal{L}}{\partial Z_2} = \frac{1}{N}(P - Y)$$

$$\frac{\partial \mathcal{L}}{\partial W_2} = H^\top \frac{\partial \mathcal{L}}{\partial Z_2},\qquad \frac{\partial \mathcal{L}}{\partial \mathbf{b}_2} = \sum_{i} \left(\frac{\partial \mathcal{L}}{\partial Z_2}\right)_{i}$$

$$\frac{\partial \mathcal{L}}{\partial H} = \frac{\partial \mathcal{L}}{\partial Z_2} W_2^\top,\qquad \frac{\partial \mathcal{L}}{\partial Z_1} = \frac{\partial \mathcal{L}}{\partial H} \odot \mathbb{1}[Z_1 > 0]$$

$$\frac{\partial \mathcal{L}}{\partial W_1} = X^\top \frac{\partial \mathcal{L}}{\partial Z_1},\qquad \frac{\partial \mathcal{L}}{\partial \mathbf{b}_1} = \sum_{i} \left(\frac{\partial \mathcal{L}}{\partial Z_1}\right)_{i}$$

عبارت زیبای $P - Y$ نتیجه ترکیب Softmax با Cross-Entropy است و یکی از دلایلی است که این دو همیشه با هم استفاده می‌شوند. یک قاعده کمکی برای بررسی: **ابعاد گرادیان هر پارامتر همیشه با ابعاد خود آن پارامتر برابر است.**

داده دوکلاسه غیرخطی `make_moons` را می‌سازیم:

</div>

In [ ]:
X, y = make_moons(n_samples=1000, noise=0.2, random_state=0)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
print('train:', X_train.shape, ' test:', X_test.shape)

plt.scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap='coolwarm', s=12)
plt.title('make_moons'); plt.show()

In [ ]:
def relu(z):
    return np.maximum(0, z)


def softmax(z):
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)


def one_hot(y, k):
    return np.eye(k)[y]


class TwoLayerNet:
    def __init__(self, n_in, n_hidden, n_out, seed=0):
        rng = np.random.default_rng(seed)
        self.params = {
            'W1': rng.normal(0, np.sqrt(2 / n_in), (n_in, n_hidden)), 'b1': np.zeros(n_hidden),
            'W2': rng.normal(0, np.sqrt(2 / n_hidden), (n_hidden, n_out)), 'b2': np.zeros(n_out),
        }

    def forward(self, X):
        p = self.params
        Z1 = X @ p['W1'] + p['b1']
        H = relu(Z1)
        Z2 = H @ p['W2'] + p['b2']
        P = softmax(Z2)
        self.cache = (X, Z1, H, P)          # stored for the backward pass
        return P

    def loss(self, P, y):
        return -np.mean(np.log(P[np.arange(len(y)), y] + 1e-12))

    def backward(self, y):
        X, Z1, H, P = self.cache
        N = len(y)
        dZ2 = (P - one_hot(y, P.shape[1])) / N
        grads = {'W2': H.T @ dZ2, 'b2': dZ2.sum(axis=0)}
        dH = dZ2 @ self.params['W2'].T
        dZ1 = dH * (Z1 > 0)
        grads['W1'] = X.T @ dZ1
        grads['b1'] = dZ1.sum(axis=0)
        return grads

<div dir="rtl">

### بررسی درستی گرادیان (Gradient Checking)

پیاده‌سازی دستی Backpropagation مستعد خطا است. برای اطمینان، گرادیان تحلیلی را با تقریب عددی **تفاضل مرکزی (Central Difference)** مقایسه می‌کنیم:

$$\frac{\partial \mathcal{L}}{\partial \theta} \approx \frac{\mathcal{L}(\theta + \epsilon) - \mathcal{L}(\theta - \epsilon)}{2\epsilon}$$

**خطای نسبی (Relative Error)** کمتر از حدود $10^{-6}$ نشان می‌دهد پیاده‌سازی درست است.

</div>

In [ ]:
def numerical_grad(net, X, y, name, eps=1e-5):
    theta = net.params[name]
    grad = np.zeros_like(theta)
    it = np.nditer(theta, flags=['multi_index'])
    for _ in it:
        idx = it.multi_index
        old = theta[idx]
        theta[idx] = old + eps; plus = net.loss(net.forward(X), y)
        theta[idx] = old - eps; minus = net.loss(net.forward(X), y)
        theta[idx] = old
        grad[idx] = (plus - minus) / (2 * eps)
    return grad


check_net = TwoLayerNet(2, 5, 2, seed=1)
Xs, ys = X_train[:20], y_train[:20]
check_net.forward(Xs)
analytic = check_net.backward(ys)
for name in ['W1', 'b1', 'W2', 'b2']:
    numeric = numerical_grad(check_net, Xs, ys, name)
    rel_err = np.abs(analytic[name] - numeric).max() / (np.abs(analytic[name]).max() + np.abs(numeric).max())
    print(f'{name}: relative error = {rel_err:.2e}')

<div dir="rtl">

## ۶. آموزش با Mini-batch Gradient Descent

سه نسخه از گرادیان کاهشی داریم:

| نسخه | داده در هر به‌روزرسانی | ویژگی |
|---|---|---|
| Batch GD | کل داده | دقیق اما کند و پرحافظه |
| Stochastic GD | یک نمونه | سریع اما بسیار پرنویز |
| **Mini-batch GD** | یک دسته کوچک (مثلاً ۳۲ تا ۲۵۶) | تعادل خوب؛ مناسب پردازش موازی روی GPU |

در عمل همیشه از Mini-batch استفاده می‌شود (و در اصطلاح رایج به آن هم SGD می‌گویند). حلقه آموزش زیر الگوی همه حلقه‌های آموزش این دوره است: **Forward → Loss → Backward → Update**.

</div>

In [ ]:
def accuracy(net, X, y):
    return np.mean(net.forward(X).argmax(axis=1) == y)


net = TwoLayerNet(n_in=2, n_hidden=32, n_out=2)
lr, epochs, batch_size = 0.5, 200, 64
history = {'loss': [], 'train_acc': [], 'test_acc': []}

for epoch in range(epochs):
    perm = np.random.permutation(len(X_train))
    epoch_loss = 0.0
    for start in range(0, len(X_train), batch_size):
        idx = perm[start:start + batch_size]
        P = net.forward(X_train[idx])                         # 1. forward
        epoch_loss += net.loss(P, y_train[idx]) * len(idx)    # 2. loss
        grads = net.backward(y_train[idx])                    # 3. backward
        for k in net.params:                                  # 4. update
            net.params[k] -= lr * grads[k]
    history['loss'].append(epoch_loss / len(X_train))
    history['train_acc'].append(accuracy(net, X_train, y_train))
    history['test_acc'].append(accuracy(net, X_test, y_test))
    if epoch % 40 == 0 or epoch == epochs - 1:
        print(f"epoch {epoch:3d}  loss {history['loss'][-1]:.4f}  "
              f"train acc {history['train_acc'][-1]:.3f}  test acc {history['test_acc'][-1]:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.2))
axes[0].plot(history['loss']); axes[0].set_title('training loss'); axes[0].set_xlabel('epoch')
axes[1].plot(history['train_acc'], label='train'); axes[1].plot(history['test_acc'], label='test')
axes[1].set_title('accuracy'); axes[1].set_xlabel('epoch'); axes[1].legend()

xx, yy = np.meshgrid(np.linspace(-2, 3, 300), np.linspace(-1.5, 2, 300))
zz = net.forward(np.c_[xx.ravel(), yy.ravel()])[:, 1].reshape(xx.shape)
axes[2].contourf(xx, yy, zz, levels=20, cmap='coolwarm', alpha=0.6)
axes[2].scatter(X_test[:, 0], X_test[:, 1], c=y_test, cmap='coolwarm', s=10, edgecolor='k', linewidth=0.3)
axes[2].set_title('decision surface (test set)')
plt.show()

<div dir="rtl">

شبکه دولایه ما، فقط با NumPy و چند خط فرمول، یک مرز تصمیم غیرخطی را یاد گرفت.

## ۷. محوشدگی گرادیان (Vanishing Gradient)

در Backpropagation، گرادیان هر لایه حاصل‌ضرب مشتق‌های لایه‌های بالاتر است. اگر مشتق تابع فعال‌سازی کوچک باشد (بیشینه مشتق Sigmoid برابر $0.25$ است)، در یک شبکه عمیق این حاصل‌ضرب به‌صورت نمایی کوچک می‌شود و **لایه‌های اول عملاً آموزش نمی‌بینند**. این مشکل برای سال‌ها مانع آموزش شبکه‌های عمیق بود.

در آزمایش زیر یک شبکه ۱۰ لایه می‌سازیم و نُرم گرادیان رسیده به هر لایه را برای Sigmoid و ReLU مقایسه می‌کنیم:

</div>

In [ ]:
def layer_gradient_norms(activation, n_layers=10, width=64, seed=0):
    rng = np.random.default_rng(seed)
    act, d_act = {
        'sigmoid': (lambda z: 1 / (1 + np.exp(-z)), lambda z: (1 / (1 + np.exp(-z))) * (1 - 1 / (1 + np.exp(-z)))),
        'relu': (lambda z: np.maximum(0, z), lambda z: (z > 0).astype(float)),
    }[activation]
    gain = 1.0 if activation == 'sigmoid' else 2.0
    Ws = [rng.normal(0, np.sqrt(gain / width), (width, width)) for _ in range(n_layers)]
    h = rng.normal(size=(128, width))
    zs = []
    for W in Ws:                         # forward pass
        z = h @ W
        zs.append(z)
        h = act(z)
    grad = np.ones_like(h)               # pretend dL/dh_out = 1
    norms = []
    for W, z in zip(reversed(Ws), reversed(zs)):   # backward pass
        grad = grad * d_act(z)
        norms.append(np.linalg.norm(grad @ W.T))
        grad = grad @ W.T
    return norms[::-1]                   # layer 1 first


layers = np.arange(1, 11)
plt.figure(figsize=(8, 4))
plt.semilogy(layers, layer_gradient_norms('sigmoid'), 'o-', label='Sigmoid')
plt.semilogy(layers, layer_gradient_norms('relu'), 's-', label='ReLU')
plt.xlabel('layer (1 = closest to the input)'); plt.ylabel('gradient norm (log scale)')
plt.title('Gradient flowing back through a 10-layer network'); plt.legend(); plt.show()

<div dir="rtl">

با Sigmoid، گرادیان در هر لایه چند برابر کوچک می‌شود و به لایه اول تقریباً چیزی نمی‌رسد؛ با ReLU (و مقداردهی اولیه مناسب) گرادیان تقریباً ثابت می‌ماند. راه‌حل‌های دیگر این مشکل را در جلسات بعد می‌بینیم: **مقداردهی اولیه (Initialization)** مناسب و **Batch Normalization** (جلسات ۵ و ۶) و **اتصال‌های میان‌بر (Residual Connections)** در ResNet (جلسه ۸). پدیده معکوس، یعنی بزرگ شدن نمایی گرادیان، **انفجار گرادیان (Exploding Gradient)** نام دارد و در RNNها (جلسه ۱۰) مهم است.

## ۸. جمع‌بندی

- تابع هزینه خطا را به یک عدد تبدیل می‌کند: MSE برای رگرسیون و Cross-Entropy برای طبقه‌بندی.
- گرادیان کاهشی خلاف جهت گرادیان قدم برمی‌دارد و نرخ یادگیری مهم‌ترین ابرپارامتر آن است.
- Backpropagation همان قاعده زنجیره‌ای روی گراف محاسباتی است: هر گره مشتق محلی خود را در گرادیان بالادستی ضرب می‌کند.
- ترکیب Softmax و Cross-Entropy گرادیان ساده $P - Y$ را می‌دهد.
- با Gradient Checking درستی پیاده‌سازی را بررسی می‌کنیم.
- Mini-batch GD استاندارد عملی است و حلقه آموزش همیشه Forward → Loss → Backward → Update است.
- در شبکه‌های عمیق با Sigmoid، گرادیان محو می‌شود؛ ReLU این مشکل را کاهش می‌دهد.

## ۹. تمرین

۱. به کلاس `Value` عملیات `tanh` و `exp` را اضافه کنید و با یک مثال، گرادیان را با مقدار تحلیلی مقایسه کنید.

۲. در `TwoLayerNet` تابع ReLU را با Tanh جایگزین کنید (مشتق: $1-\tanh^2$). گرادیان را با Gradient Checking بررسی کنید و سرعت همگرایی را مقایسه کنید.

۳. حلقه آموزش را با `batch_size` برابر ۱، ۶۴ و کل داده اجرا کنید و نمودار Loss را مقایسه کنید. کدام سریع‌تر (بر حسب زمان واقعی) همگرا می‌شود؟

۴. یک لایه پنهان دوم به شبکه اضافه کنید (`TwoLayerNet` را به `ThreeLayerNet` تبدیل کنید) و فرمول‌های Backward را برای آن بنویسید.

</div>